# CONDOR Set-Transformer Training for FlexDC Behavior Labels

This notebook is based on the latest configured-objective CONDOR/FlexDC training notebook, but changes the **direct neural labels** while preserving the CONDOR training framework and Set-Transformer workload encoder.

The network now directly predicts:

1. `log(mean normalized tracking error + 1e-6)`
2. `log(p90 normalized tracking error + 0.001)`
3. one QoS violation probability `P_j` for every real job type

It does **not** directly predict `M_RSR`, `Ctrack`, `CQoS`, or the full objective. Those are reconstructed from the predicted raw behavior using the known FlexDC equations.

The model and data loader support variable-length workload mixes using batch padding plus masks. The current sweep contains only `J=4`; this run tests the new labels on the current pilot data and does not claim unseen-`J` generalization.

W&B logs, at every epoch:

- total and component losses;
- tracking, QoS, reconstructed-cost, and objective metrics;
- overall feasible/infeasible accuracy;
- accuracy on **actual feasible rows**;
- accuracy on **actual infeasible rows**;
- false-feasible and false-infeasible rates.

## 0. Choose environment — RUN EVERYWHERE

Use `colab` in Google Colab. Use `local_pc` only when the repository and dataset already exist locally.

In [ ]:
from pathlib import Path
import os
import sys

RUN_ENV = "colab"  # "colab" or "local_pc"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

# Repository settings.
COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC.git"
COMDER_BRANCH = "main"
WORKSPACE = Path("/content/workspace") if RUN_ENV == "colab" else Path.cwd().parent

# W&B settings.
USE_WANDB = True
WANDB_MODE = "online"  # "online", "offline", or "disabled"
WANDB_ENTITY = "amenon06-boston-university"
WANDB_PROJECT = "flexdc-unified-training"

print("RUN_ENV:", RUN_ENV)
print("Detected Colab-like environment:", IN_COLAB)
print("WORKSPACE:", WORKSPACE)
print("USE_WANDB:", USE_WANDB, "WANDB_MODE:", WANDB_MODE)

## 1. Install dependencies — COLAB ONLY

Do not run this locally unless you intentionally want to install into the active environment.

In [ ]:
import subprocess
import sys

if RUN_ENV == "colab":
    packages = [
        "wandb",
        "pandas",
        "numpy",
        "scipy",
        "scikit-learn",
        "tqdm",
        "matplotlib",
        "tabulate",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("SKIP: local_pc mode")

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Clone or update the CONDOR-FLEXDC repository — COLAB ONLY

Opening the notebook from GitHub does not clone the rest of the repository into the Colab runtime.

In [ ]:
import subprocess

if RUN_ENV == "colab":
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    COMDER_ROOT = WORKSPACE / "comder-main"
    if not COMDER_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--branch", COMDER_BRANCH,
            COMDER_REPO_URL, str(COMDER_ROOT),
        ])
    else:
        subprocess.check_call(["git", "fetch"], cwd=str(COMDER_ROOT))
        subprocess.check_call(["git", "checkout", COMDER_BRANCH], cwd=str(COMDER_ROOT))
        subprocess.check_call(["git", "pull"], cwd=str(COMDER_ROOT))
    print("COMDER_ROOT:", COMDER_ROOT)
else:
    print("SKIP: local_pc mode. Set COMDER_ROOT in the local-path cell.")

## 3. Optional Google Drive copy — COLAB ONLY

Run this only when the new CSVs/checkpoint support files are not committed to GitHub. The notebook expects the two paper-objective CSVs produced from the completed Sweep V2 run.

In [ ]:
USE_GOOGLE_DRIVE_DATA = False

# Edit only if USE_GOOGLE_DRIVE_DATA=True.
DRIVE_RESULTS_CSV = "/content/drive/MyDrive/path/to/flexdc_sweep_combined_grid_search_results.csv"
DRIVE_DIAGNOSTICS_CSV = "/content/drive/MyDrive/path/to/flexdc_sweep_combined_grid_search_diagnostics.csv"
DRIVE_MODEL_PY = "/content/drive/MyDrive/path/to/data_center_model_flexdc_behavior.py"
DRIVE_UTILS_PY = "/content/drive/MyDrive/path/to/am_flexdc_behavior_training_utilities.py"
DRIVE_TEST_PY = "/content/drive/MyDrive/path/to/test_flexdc_behavior_training_v1.py"

if RUN_ENV == "colab" and USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
    TRAIN_DIR = AM_FLEXDC_ROOT / "train"
    PILOT_DIR = AM_FLEXDC_ROOT / "data" / "pilots" / "flexdc_sweep_v2_paper_objective"
    PILOT_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copy2(DRIVE_RESULTS_CSV, PILOT_DIR / "flexdc_sweep_combined_grid_search_results.csv")
    shutil.copy2(DRIVE_DIAGNOSTICS_CSV, PILOT_DIR / "flexdc_sweep_combined_grid_search_diagnostics.csv")
    shutil.copy2(DRIVE_MODEL_PY, TRAIN_DIR / "data_center_model_flexdc_behavior.py")
    shutil.copy2(DRIVE_UTILS_PY, TRAIN_DIR / "am_flexdc_behavior_training_utilities.py")
    shutil.copy2(DRIVE_TEST_PY, TRAIN_DIR / "test_flexdc_behavior_training_v1.py")
    print("Copied dataset and behavior-training support files from Drive.")
else:
    print("Drive copy skipped.")

## 4. Local Windows paths — LOCAL PC ONLY

In [ ]:
if RUN_ENV == "local_pc":
    COMDER_ROOT = Path(r"C:\Users\Achuthan Menon\Desktop\Research Work\comder-main")
    if not COMDER_ROOT.exists():
        raise FileNotFoundError(f"COMDER_ROOT does not exist: {COMDER_ROOT}")
    print("Local COMDER_ROOT:", COMDER_ROOT)
else:
    print("SKIP: Colab mode")

## 5. Resolve paths and check all required files — RUN EVERYWHERE

The new model and utility files are intentionally separate from the previous three-output model so historical checkpoints remain usable.

In [ ]:
AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
TRAIN_DIR = AM_FLEXDC_ROOT / "train"
MODELS_DIR = AM_FLEXDC_ROOT / "models" / "flexdc_behavior"
RESULTS_DIR = AM_FLEXDC_ROOT / "results" / "training_runs"
PILOT_DIR = AM_FLEXDC_ROOT / "data" / "pilots" / "flexdc_sweep_v2_paper_objective"

RESULTS_CSV = PILOT_DIR / "flexdc_sweep_combined_grid_search_results.csv"
DIAGNOSTICS_CSV = PILOT_DIR / "flexdc_sweep_combined_grid_search_diagnostics.csv"

MODEL_PY = TRAIN_DIR / "data_center_model_flexdc_behavior.py"
UTILS_PY = TRAIN_DIR / "am_flexdc_behavior_training_utilities.py"
TEST_PY = TRAIN_DIR / "test_flexdc_behavior_training_v1.py"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required = [MODEL_PY, UTILS_PY, TEST_PY, RESULTS_CSV, DIAGNOSTICS_CSV]
missing = [path for path in required if not path.exists()]
if missing:
    print("Missing required files:")
    for path in missing:
        print(" -", path)
    raise FileNotFoundError("Fix missing paths before training.")

print("All required files found.")
print("TRAIN_DIR:", TRAIN_DIR)
print("RESULTS_CSV:", RESULTS_CSV)
print("DIAGNOSTICS_CSV:", DIAGNOSTICS_CSV)
print("MODELS_DIR:", MODELS_DIR)

## 6. Compile and run structural tests — RUN BEFORE TRAINING

These tests check variable-length batching, padding masks, job-order invariance, per-job output ordering, and zero gradient through padded jobs.

In [ ]:
import subprocess
import sys

os.chdir(TRAIN_DIR)
compile_cmd = [
    sys.executable,
    "-m",
    "py_compile",
    MODEL_PY.name,
    UTILS_PY.name,
    TEST_PY.name,
]
print("Compile command:", " ".join(compile_cmd))
subprocess.run(compile_cmd, check=True)
subprocess.run([sys.executable, TEST_PY.name], check=True)

## 7. Import the adapted CONDOR model and training utilities

In [ ]:
if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))
os.chdir(TRAIN_DIR)

from data_center_model_flexdc_behavior import (
    DataCenterBehaviorModel,
    FlexDCBehaviorModelConfig,
)
from am_flexdc_behavior_training_utilities import (
    FlexDCBehaviorConstants,
    final_behavior_evaluation,
    prepare_behavior_data,
    sample_prediction_rows,
    save_behavior_checkpoint,
    train_behavior_model,
)

print("Imports successful.")

## 8. W&B login

Set `USE_WANDB=False` to train without W&B. In Colab, the cell first checks a Colab secret named `WANDB_API_KEY`, then falls back to interactive login.

In [ ]:
if USE_WANDB and WANDB_MODE != "disabled":
    import getpass
    import wandb

    os.environ["WANDB_MODE"] = WANDB_MODE
    os.environ.pop("WANDB_BASE_URL", None)
    login_ok = False

    if RUN_ENV == "colab":
        try:
            from google.colab import userdata
            secret_key = userdata.get("WANDB_API_KEY")
        except Exception:
            secret_key = None
        if secret_key:
            os.environ["WANDB_API_KEY"] = secret_key
            login_ok = wandb.login(key=secret_key, relogin=True, verify=True)

    if not login_ok and WANDB_MODE == "online":
        try:
            login_ok = wandb.login(relogin=True, verify=True)
        except Exception as exc:
            print("Automatic W&B login did not complete:", repr(exc))
            key = getpass.getpass("Paste W&B API key: ").strip()
            os.environ["WANDB_API_KEY"] = key
            login_ok = wandb.login(key=key, relogin=True, verify=True)

    if WANDB_MODE == "online" and not login_ok:
        raise RuntimeError("W&B online login failed. Set USE_WANDB=False or WANDB_MODE='offline'.")
    print("W&B ready; mode:", WANDB_MODE)
else:
    wandb = None
    print("W&B disabled.")

## 9. Training configuration — EDIT/RUN

This first experiment isolates the label redesign. Loss-component weights are equal and no additional boundary weighting or direct objective loss is enabled.

In [ ]:
# Run identity.
DATASET_TAG = "flexdc_sweep_v2_behavior_labels_v1"
RUN_NAME = f"condor_set_transformer_{DATASET_TAG}"

# Data/split settings.
USE_NORM_PR = True
USE_NORM_WLMIX = True
DEDUPLICATE = True
HELDOUT_FRACTION = 0.30
SPLIT_SEED = 0
NUM_WORKERS = 0  # safest in Colab notebooks

# Training settings.
EPOCHS = 150
LR = 1e-4
BATCH_SIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METRICS_EVERY_N_EPOCHS = 1

# Equal first-pass loss weights.
MEAN_TRACKING_LOSS_WEIGHT = 1.0
P90_TRACKING_LOSS_WEIGHT = 1.0
QOS_LOSS_WEIGHT = 1.0

# Exact FlexDC paper constants used for analytic reconstruction.
BEHAVIOR_CONSTANTS = FlexDCBehaviorConstants(
    mean_tracking_log_floor=1e-6,
    p90_tracking_log_floor=1e-3,
    tracking_threshold=0.3,
    qos_threshold=0.1,
    ctrack_psi=1.0,
    ctrack_mu=10.0,
    qos_beta=20.0,
    qos_rho=2.0,
)

# Preserve the CONDOR Set-Transformer dimensions while changing the output heads.
MODEL_CONFIG = FlexDCBehaviorModelConfig(
    dim_job_mix=7,
    dim_dc_features=5,
    st_dim_hidden=512,
    st_dim_output=1019,
    st_num_heads=4,
    st_num_outputs=1,
    linear_dim_hidden=512,
    qos_projection_dim=256,
    skip_connections=True,
    layer_norm=False,
)

MODEL_FILE = MODELS_DIR / f"am_{DATASET_TAG}_checkpoint.pt"
METRICS_FILE = RESULTS_DIR / f"am_{DATASET_TAG}_metrics.csv"
HISTORY_FILE = RESULTS_DIR / f"am_{DATASET_TAG}_history.csv"
TRAIN_PREDICTIONS_FILE = RESULTS_DIR / f"am_{DATASET_TAG}_train_predictions.csv"
HELDOUT_PREDICTIONS_FILE = RESULTS_DIR / f"am_{DATASET_TAG}_heldout_predictions.csv"
DATA_AUDIT_FILE = RESULTS_DIR / f"am_{DATASET_TAG}_data_audit.json"

print("Run:", RUN_NAME)
print("Device:", DEVICE)
print("Model checkpoint:", MODEL_FILE)

## 10. Prepare and audit the dataset — RUN BEFORE TRAINING

This cell:

- merges results and diagnostics by `Plan_Row_ID`;
- removes the 39 redundant duplicate configurations after verifying their labels agree;
- keeps seed replicates in the same side of the 70/30 split using `Base_Plan_Row_ID`;
- computes workload normalization and log-label statistics from the training split only;
- pads workload mixes only within each batch and creates masks.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

behavior_data = prepare_behavior_data(
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    batch_size=BATCH_SIZE,
    heldout_fraction=HELDOUT_FRACTION,
    split_seed=SPLIT_SEED,
    use_norm_pr=USE_NORM_PR,
    use_norm_wlmix=USE_NORM_WLMIX,
    num_workers=NUM_WORKERS,
    deduplicate=DEDUPLICATE,
    constants=BEHAVIOR_CONSTANTS,
)

DATA_AUDIT_FILE.write_text(json.dumps(behavior_data.audit, indent=2))

display(Markdown("### Data audit"))
display(pd.DataFrame([behavior_data.audit]).T.rename(columns={0: "Value"}))

# Physical feasibility distribution in the two split datasets.
def split_feasibility_table(dataset, split_name):
    frame = dataset.df.copy()
    max_pj = frame["QoS_Delay_Probabilities"].apply(
        lambda x: max(json.loads(x) if isinstance(x, str) else x)
    )
    tracking_pass = frame["Ctrack_Epsilon_90th"].astype(float) <= BEHAVIOR_CONSTANTS.tracking_threshold
    qos_pass = max_pj <= BEHAVIOR_CONSTANTS.qos_threshold
    both = tracking_pass & qos_pass
    return {
        "Split": split_name,
        "Rows": len(frame),
        "Tracking pass": int(tracking_pass.sum()),
        "QoS pass": int(qos_pass.sum()),
        "Both pass": int(both.sum()),
        "Both pass %": 100.0 * both.mean(),
        "Both fail": int((~tracking_pass & ~qos_pass).sum()),
    }

split_summary = pd.DataFrame([
    split_feasibility_table(behavior_data.train_dataset, "Train"),
    split_feasibility_table(behavior_data.heldout_dataset, "Heldout"),
])
display(Markdown("### Feasibility distribution"))
display(split_summary.style.hide(axis="index").format({"Both pass %": "{:.2f}%"}))

print("Direct labels:", behavior_data.metadata.direct_label_names)
print("Workload sizes present:", behavior_data.audit["workload_sizes"])
print("Log mean-tracking train mean/std:", behavior_data.metadata.log_mean_tracking_mean, behavior_data.metadata.log_mean_tracking_std)
print("Log p90 train mean/std:", behavior_data.metadata.log_p90_tracking_mean, behavior_data.metadata.log_p90_tracking_std)
print("Saved audit:", DATA_AUDIT_FILE)

## 11. Create the adapted CONDOR model

In [ ]:
model = DataCenterBehaviorModel(MODEL_CONFIG)
print(model)
print("Trainable parameters:", f"{model.parameter_count():,}")

# One forward-pass shape check using a real batch.
example_batch = next(iter(behavior_data.train_loader))
with torch.no_grad():
    example_output = model(
        example_batch["features"],
        example_batch["workload"],
        example_batch["mask"],
    )
print("Tracking-log output shape:", tuple(example_output["tracking_logs"].shape))
print("Per-job QoS output shape:", tuple(example_output["qos_probabilities"].shape))
print("Real-job counts in example batch:", example_batch["mask"].sum(dim=1)[:10].tolist())

## 12. Train one model with W&B metrics

The W&B feasibility metrics are logged every epoch under:

```text
train/feasibility/combined/accuracy_overall
train/feasibility/combined/actual_feasible_accuracy
train/feasibility/combined/actual_infeasible_accuracy
heldout/feasibility/combined/accuracy_overall
heldout/feasibility/combined/actual_feasible_accuracy
heldout/feasibility/combined/actual_infeasible_accuracy
```

Separate tracking and QoS feasibility metrics are logged under parallel paths.

In [ ]:
training_config = {
    "dataset_tag": DATASET_TAG,
    "direct_labels": behavior_data.metadata.direct_label_names,
    "architecture": "CONDOR masked Set Transformer + two tracking heads + shared per-job QoS head",
    "model_config": MODEL_CONFIG.to_dict(),
    "data_metadata": behavior_data.metadata.to_dict(),
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "device": DEVICE,
    "metrics_every_n_epochs": METRICS_EVERY_N_EPOCHS,
    "mean_tracking_loss_weight": MEAN_TRACKING_LOSS_WEIGHT,
    "p90_tracking_loss_weight": P90_TRACKING_LOSS_WEIGHT,
    "qos_loss_weight": QOS_LOSS_WEIGHT,
}

run = None
if USE_WANDB and WANDB_MODE != "disabled":
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=RUN_NAME,
        config=training_config,
        tags=["condor", "set-transformer", "flexdc", "raw-behavior-labels", "per-job-qos"],
        mode=WANDB_MODE,
        save_code=True,
    )

model, history = train_behavior_model(
    model,
    behavior_data,
    epochs=EPOCHS,
    lr=LR,
    device_name=DEVICE,
    mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
    p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
    qos_weight=QOS_LOSS_WEIGHT,
    wandb_run=run,
    metrics_every_n_epochs=METRICS_EVERY_N_EPOCHS,
    verbose=True,
)

history.to_csv(HISTORY_FILE, index=False)
print("Saved history:", HISTORY_FILE)

## 13. Final evaluation, formatted tables, checkpoint, and W&B prediction samples

In [ ]:
metrics, train_predictions, heldout_predictions = final_behavior_evaluation(
    model,
    behavior_data,
    device_name=DEVICE,
    mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
    p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
    qos_weight=QOS_LOSS_WEIGHT,
)

pd.DataFrame([metrics]).to_csv(METRICS_FILE, index=False)
train_predictions.to_csv(TRAIN_PREDICTIONS_FILE, index=False)
heldout_predictions.to_csv(HELDOUT_PREDICTIONS_FILE, index=False)

save_behavior_checkpoint(
    MODEL_FILE,
    model=model,
    data_metadata=behavior_data.metadata,
    model_config=MODEL_CONFIG.to_dict(),
    training_config=training_config,
    metrics=metrics,
)

key_metrics = {
    "Heldout total loss": metrics["heldout/loss/total"],
    "Heldout mean tracking MAE": metrics["heldout/tracking/mean_physical/mae"],
    "Heldout p90 tracking MAE": metrics["heldout/tracking/p90_physical/mae"],
    "Heldout per-job Pj MAE": metrics["heldout/qos/per_job_probability/mae"],
    "Heldout max Pj MAE": metrics["heldout/qos/max_probability/mae"],
    "Heldout M_RSR MAE": metrics["heldout/cost/M_RSR/mae"],
    "Heldout objective MAE": metrics["heldout/cost/full_objective/mae"],
    "Heldout objective R2": metrics["heldout/cost/full_objective/r2"],
    "Heldout objective Spearman": metrics["heldout/cost/full_objective/spearman"],
    "Feasibility overall accuracy": metrics["heldout/feasibility/combined/accuracy_overall"],
    "Actual feasible accuracy": metrics["heldout/feasibility/combined/actual_feasible_accuracy"],
    "Actual infeasible accuracy": metrics["heldout/feasibility/combined/actual_infeasible_accuracy"],
    "Feasible precision": metrics["heldout/feasibility/combined/feasible_precision"],
    "False feasible rate": metrics["heldout/feasibility/combined/false_feasible_rate"],
}
key_df = pd.DataFrame([{"Metric": key, "Value": value} for key, value in key_metrics.items()])
display(Markdown("### Final heldout summary"))
display(
    key_df.style.hide(axis="index")
    .format({"Value": "{:.6f}"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#1f2937"), ("color", "white"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "td", "props": [("border", "1px solid #ddd"), ("padding", "6px"), ("text-align", "left")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
    ])
)

# Confusion-matrix-style counts Fatih can inspect directly.
feasibility_counts = pd.DataFrame([
    {
        "Split": "Train",
        "Actual feasible": metrics["train/feasibility/combined/actual_feasible_count"],
        "Actual infeasible": metrics["train/feasibility/combined/actual_infeasible_count"],
        "False feasible": metrics["train/feasibility/combined/false_feasible_count"],
        "False infeasible": metrics["train/feasibility/combined/false_infeasible_count"],
        "Actual feasible accuracy": metrics["train/feasibility/combined/actual_feasible_accuracy"],
        "Actual infeasible accuracy": metrics["train/feasibility/combined/actual_infeasible_accuracy"],
    },
    {
        "Split": "Heldout",
        "Actual feasible": metrics["heldout/feasibility/combined/actual_feasible_count"],
        "Actual infeasible": metrics["heldout/feasibility/combined/actual_infeasible_count"],
        "False feasible": metrics["heldout/feasibility/combined/false_feasible_count"],
        "False infeasible": metrics["heldout/feasibility/combined/false_infeasible_count"],
        "Actual feasible accuracy": metrics["heldout/feasibility/combined/actual_feasible_accuracy"],
        "Actual infeasible accuracy": metrics["heldout/feasibility/combined/actual_infeasible_accuracy"],
    },
])
display(Markdown("### Feasible/infeasible classification summary"))
display(feasibility_counts.style.hide(axis="index").format(precision=5))

if run is not None:
    run.log(metrics)
    run.summary.update(metrics)
    train_sample = sample_prediction_rows(train_predictions, max_rows=1024, seed=0)
    heldout_sample = sample_prediction_rows(heldout_predictions, max_rows=1024, seed=1)
    run.log({
        "final/train_prediction_samples": wandb.Table(dataframe=train_sample),
        "final/heldout_prediction_samples": wandb.Table(dataframe=heldout_sample),
        "final/feasibility_summary": wandb.Table(dataframe=feasibility_counts),
    })
    run.save(str(MODEL_FILE))
    run.finish()

print("Saved checkpoint:", MODEL_FILE)
print("Saved metrics:", METRICS_FILE)
print("Saved train predictions:", TRAIN_PREDICTIONS_FILE)
print("Saved heldout predictions:", HELDOUT_PREDICTIONS_FILE)

## 14. Plot training curves — OPTIONAL

In [ ]:
import matplotlib.pyplot as plt

if len(history):
    curve_specs = [
        ("heldout/loss/total", "Heldout total loss"),
        ("heldout/tracking/p90_physical/mae", "Heldout p90 tracking MAE"),
        ("heldout/qos/max_probability/mae", "Heldout max Pj MAE"),
        ("heldout/feasibility/combined/accuracy_overall", "Heldout feasibility accuracy"),
        ("heldout/feasibility/combined/actual_feasible_accuracy", "Accuracy on actual feasible rows"),
        ("heldout/feasibility/combined/actual_infeasible_accuracy", "Accuracy on actual infeasible rows"),
    ]
    for column, title in curve_specs:
        if column not in history.columns:
            continue
        plt.figure(figsize=(8, 4))
        plt.plot(history["epoch"], history[column])
        plt.xlabel("Epoch")
        plt.ylabel(title)
        plt.title(title)
        plt.grid(alpha=0.3)
        plt.show()

## 15. Download outputs — COLAB OPTIONAL

In [ ]:
if RUN_ENV == "colab":
    from google.colab import files
    import shutil

    archive_root = RESULTS_DIR / f"{DATASET_TAG}_artifacts"
    archive_root.mkdir(parents=True, exist_ok=True)
    for source in [
        MODEL_FILE,
        METRICS_FILE,
        HISTORY_FILE,
        TRAIN_PREDICTIONS_FILE,
        HELDOUT_PREDICTIONS_FILE,
        DATA_AUDIT_FILE,
    ]:
        if source.exists():
            shutil.copy2(source, archive_root / source.name)

    zip_path = shutil.make_archive(str(archive_root), "zip", root_dir=archive_root)
    print("Created:", zip_path)
    # Uncomment to download automatically:
    # files.download(zip_path)
else:
    print("Local mode: outputs are already saved under", RESULTS_DIR)